# CrisisInbox — Minimal GRPO Training (Unsloth + OpenEnv)

Train Qwen2.5-0.5B to triage crisis inbox messages using **online GRPO rewards** from the live OpenEnv environment.

**Requirements:** Colab T4 GPU or any CUDA machine.  
**Environment:** [eptan-crisis-inbox.hf.space](https://eptan-crisis-inbox.hf.space)

In [ ]:
# Install dependencies
!pip install unsloth trl openenv-client -q

## 1. Load Model with Unsloth LoRA

In [ ]:
from unsloth import FastLanguageModel

MODEL = "unsloth/Qwen2.5-0.5B-Instruct"
MAX_SEQ = 2048
MAX_PROMPT = 1536

model, tokenizer = FastLanguageModel.from_pretrained(MODEL, max_seq_length=MAX_SEQ, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
print(f"Model loaded: {MODEL}, max_seq={MAX_SEQ}")

## 2. Collect Prompts from Live Environment

In [ ]:
from openenv import GenericEnvClient, GenericAction

BASE_URL = "https://eptan-crisis-inbox.hf.space"

SYSTEM_PROMPT = """You are a crisis inbox triage assistant. Respond using EXACTLY this format:

respond_to_message("msg_XXX", "your response here")

Pick the most urgent unhandled message. Watch deadlines. Adapt to policy changes."""


def call_tool(env, name, **args):
    return env.step(GenericAction(tool_name=name, arguments=args))


def truncate_prompt(prompt_text, tokenizer, max_tokens):
    """Left-truncate prompt to fit token budget (drops oldest messages first)."""
    ids = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt_text}],
        return_tensors="pt", add_generation_prompt=True,
    )
    if hasattr(ids, "input_ids"):
        ids = ids.input_ids
    if ids.shape[-1] <= max_tokens:
        return prompt_text
    ids = ids[:, -max_tokens:]
    return tokenizer.decode(ids[0], skip_special_tokens=True)


# Collect prompts across seeds and timesteps
prompts = []
prompt_lookup = {}

for seed in range(5):
    with GenericEnvClient(BASE_URL, message_timeout_s=60.0).sync() as env:
        env.reset(seed=seed)
        for hour in range(0, 48, 2):
            if hour > 0:
                call_tool(env, "advance_time", hours=2.0)
            result = call_tool(env, "get_prompt", format="chatml")
            prompt_text = result.observation.get("prompt", "")
            if not prompt_text.strip():
                continue

            entry = {"prompt": prompt_text, "seed": seed, "hour": hour}
            truncated = truncate_prompt(prompt_text, tokenizer, MAX_PROMPT)
            key = truncated[:200]
            prompt_lookup[key] = entry

            prompts.append({"prompt": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": truncated},
            ]})

print(f"Collected {len(prompts)} prompts from {len(set(e['seed'] for e in prompt_lookup.values()))} seeds")

## 3. Reward Function

- **Format compliance** (0–2 pts) — scored locally, graduated credit for parseable `respond_to_message()` calls
- **Triage reward** — from `env.step()` on the live environment (urgency, deadlines, drift, tone, prioritization)

In [ ]:
import re


def score_format(completion):
    """Format compliance reward (0.0 to 2.0)."""
    if not completion or not completion.strip():
        return 0.0
    s = 0.1  # Base reward for non-empty output
    if "respond_to_message" in completion: s += 0.4
    if re.search(r'msg_\d+', completion): s += 0.5
    if re.search(r'respond_to_message\s*\(', completion): s += 0.5
    if re.search(r'respond_to_message\s*\(\s*["\']?msg_\d+["\']?\s*,\s*["\']', completion): s += 0.5
    return s


def reward_fn(prompts_batch, completions, **kwargs):
    """Online GRPO reward: format (local) + triage (env.step)."""
    rewards = []
    for prompt_msgs, completion in zip(prompts_batch, completions):
        comp = completion[-1]["content"] if isinstance(completion, list) else str(completion)
        prompt_text = prompt_msgs[-1]["content"] if isinstance(prompt_msgs, list) else str(prompt_msgs)
        entry = prompt_lookup.get(prompt_text[:200])

        if entry is None:
            rewards.append(0.0)
            continue

        match = re.search(
            r'respond_to_message\s*\(\s*["\']?(msg_\d+)["\']?\s*,\s*["\'](.+?)["\']',
            comp, re.DOTALL,
        )
        if not match:
            rewards.append(score_format(comp))
            continue

        msg_id, response = match.group(1), match.group(2)

        try:
            with GenericEnvClient(BASE_URL, message_timeout_s=30.0).sync() as env:
                env.reset(seed=entry["seed"])
                remaining = entry["hour"]
                while remaining > 0.1:
                    call_tool(env, "advance_time", hours=min(remaining, 4.0))
                    remaining -= 4.0
                result = env.step(GenericAction(
                    tool_name="respond_to_message",
                    arguments={"message_id": msg_id, "response": response},
                ))
                rewards.append(score_format(comp) + float(result.reward or 0.0))
        except Exception:
            rewards.append(score_format(comp))

    return rewards


# Quick test
print("Format scores:")
print(f"  Empty:      {score_format(''):.1f}")
print(f"  Gibberish:  {score_format('hello world'):.1f}")
print(f"  Full call:  {score_format('respond_to_message(\"msg_001\", \"Evacuating now.\")'):.1f}")

## 4. Train with GRPO

In [ ]:
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

dataset = Dataset.from_list(prompts)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_fn,
    args=GRPOConfig(
        output_dir="crisisinbox-grpo",
        max_steps=200,
        per_device_train_batch_size=2,
        learning_rate=1e-6,
        max_completion_length=256,
        max_prompt_length=MAX_PROMPT,
        num_generations=16,
        generation_batch_size=16,
        logging_steps=8,
        save_steps=200,
        bf16=True,
    ),
    train_dataset=dataset,
)

print(f"Ready — {len(dataset)} prompts, online rewards via {BASE_URL}")

In [ ]:
trainer.train()
print("Training complete")

## 5. Evaluate & Save

In [ ]:
import random

FastLanguageModel.for_inference(model)

eval_prompts = random.sample(prompts, min(10, len(prompts)))
rewards = []

for p in eval_prompts:
    msgs = p["prompt"]
    inputs = tokenizer.apply_chat_template(
        msgs, return_tensors="pt", add_generation_prompt=True, return_dict=True,
    ).to(model.device)
    prompt_len = inputs.input_ids.shape[1]

    outputs = model.generate(
        **inputs, max_new_tokens=256, temperature=0.7, do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
    )
    completion = tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True)

    r = reward_fn([msgs], [completion])[0]
    rewards.append(r)
    print(f"  Reward {r:.2f} | {completion[:80]}")

print(f"\nAvg reward: {sum(rewards)/len(rewards):.2f}")

In [ ]:
model.save_pretrained("crisisinbox-grpo-trained")
tokenizer.save_pretrained("crisisinbox-grpo-trained")
print("Model saved to crisisinbox-grpo-trained/")